# Finetuning for speaker-diarization-3.1

**Já existe um dataset pronto**

Prepare o dataset seguindo o notebook `1_dataset-preparation.ipynb`.
Após a preparação, a estrutura esperada é:
- `dataset/audios/` — arquivos `.wav`
- `dataset/rttms/{train,dev,test}/` — arquivos `.rttm`
- `dataset/uems/{train,dev,test}/` — arquivos `.uem`
- `dataset/lists/` — `train.txt`, `dev.txt`, `test.txt`


# FInetuning using diarizers

In [ ]:
HF_TOKEN = ""

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python3 diarizers/train_segmentation.py \
  --dataset_name=${HF_DATASET_NAME:-seu-usuario/nome-do-dataset} \
  --split_on_subset=train \
  --model_name_or_path=pyannote/segmentation-3.0 \
  --output_dir=./speaker-segmentation-finetuned \
  --do_train --do_eval \
  --num_train_epochs=15 \
  --per_device_train_batch_size=16 \
  --per_device_eval_batch_size=16 \
  --learning_rate=3e-3 \
  --lr_scheduler_type=linear \
  --warmup_ratio=0.1 \
  --eval_strategy=epoch --save_strategy=epoch \
  --load_best_model_at_end \
  --metric_for_best_model=eval_der --greater_is_better=false \
  --save_total_limit=2 \
  --max_grad_norm=1.0 \
  --preprocessing_num_workers=2 --dataloader_num_workers=2 \
  --logging_steps=100


In [ ]:
!python3 diarizers/train_segmentation.py \
  --dataset_name=${HF_DATASET_NAME:-seu-usuario/nome-do-dataset} \
  --split_on_subset=train \
  --model_name_or_path=pyannote/segmentation-3.0 \
  --output_dir=./speaker-segmentation-finetuned \
  --do_train --do_eval \
  --num_train_epochs=20 \
  --per_device_train_batch_size=32 \
  --per_device_eval_batch_size=32 \
  --learning_rate=3e-3 \
  --lr_scheduler_type=reduce_lr_on_plateau \
  --lr_scheduler_kwargs='{"mode":"min","factor":0.5,"patience":1,"threshold":0.0,"cooldown":0}' \
  --warmup_ratio=0.04 \
  --weight_decay=0.03 \
  --max_grad_norm=1.0 \
  --eval_strategy=epoch --save_strategy=epoch \
  --metric_for_best_model=eval_der --greater_is_better=false \
  --load_best_model_at_end --save_total_limit=2 \
  --preprocessing_num_workers=2 --dataloader_num_workers=2 \
  --logging_steps=100 --fp16


In [ ]:
!python diarizers/test_segmentation.py \
    --dataset_name=${HF_DATASET_NAME:-seu-usuario/nome-do-dataset} \
    --split_on_subset=test \
    --test_split_name=test \
    --model_name_or_path=${HF_FINETUNED_MODEL:-seu-usuario/segmentation-finetuned} \
    --preprocessing_num_workers=2 \
    --evaluate_with_pipeline